<div style='text-align: center; padding: 30px; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); border-radius: 15px; margin: 10px 0; box-shadow: 0 10px 30px rgba(0,0,0,0.2);'>
  <h1 style='color: white; margin: 0 0 8px 0; font-size: 2.5em;'>🎵 AceStep 1.5 XL Turbo (4B)</h1>
  <h3 style='color: #f0f0f0; margin: 0 0 5px 0; font-weight: 400;'>Google Colab GPU Edition — Created by <strong>AIQUEST</strong></h3>
  <p style='color: #ddd; margin: 0; text-align: center;'>Wan2GP Engine + mmgp Offloading | T4/L4 GPU | INT8 Quantized</p>
</div>

---

### Quick Start in Google Colab
1. Open **Runtime → Change runtime type → T4 GPU** (a single GPU is sufficient).
2. Make sure the runtime has Internet access.
3. Run the cells in order. The model download is about 7 GB and can take a while.
4. Open the **public Gradio link** printed by the final cell.

The notebook stores the repository and checkpoints under `/content`. Colab runtimes are temporary, so repeat the setup after a runtime reset. If you mount Google Drive, you can change `COLAB_ROOT` in Cell 1, but local `/content` is usually faster for inference.

| Feature | Details |
|---|---|
| Model | AceStep 1.5 XL Turbo 4B (quanto int8) |
| Engine | Wan2GP + mmgp Profile 2 |
| GPU | T4/L4 or another CUDA GPU with sufficient VRAM |
| Inference Steps | 8 (distilled turbo) |
| Mode | Text (Lyrics) → Music |

## ⚙️ Cell 1 — Environment Setup

In [1]:
# Cell 1: Environment Setup (Google Colab)
import gc
import os
import subprocess
import psutil

COLAB_ROOT = os.environ.get("AUDIOGEN_ROOT", "/content")
WAN2GP_DIR = os.path.join(COLAB_ROOT, "Wan2GP")
MODEL_DIR = os.path.join(WAN2GP_DIR, "models")
os.environ["WAN2GP_DIR"] = WAN2GP_DIR
os.makedirs(COLAB_ROOT, exist_ok=True)

print("=== Colab GPU Environment Setup ===")
print(f"Working directory: {COLAB_ROOT}")
print(f"RAM: {psutil.virtual_memory().total / 1024**3:.1f} GB total, {psutil.virtual_memory().available / 1024**3:.1f} GB available")

# These settings are portable to Colab; sudo-based kernel-cache tuning from Kaggle
# is intentionally omitted because it is unavailable/restricted in many runtimes.
gc.collect()
os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,garbage_collection_threshold:0.6"
os.environ["MALLOC_TRIM_THRESHOLD_"] = "0"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

try:
    subprocess.run(["nvidia-smi"], check=True)
    print("✅ GPU active")
except (FileNotFoundError, subprocess.CalledProcessError):
    raise RuntimeError("No CUDA GPU detected. In Colab select Runtime → Change runtime type → T4 GPU, then rerun this cell.")

print(f"Wan2GP directory: {WAN2GP_DIR}")
print("✅ Environment ready")


=== Colab GPU Environment Setup ===
Working directory: /content
RAM: 12.7 GB total, 11.7 GB available
✅ GPU active
Wan2GP directory: /content/Wan2GP
✅ Environment ready


## 📦 Cell 2 — Install Dependencies

In [2]:
# Cell 2: Clone Wan2GP & Install Dependencies
import os
import subprocess
import sys

COLAB_ROOT = os.environ.get("AUDIOGEN_ROOT", "/content")
WAN2GP_DIR = os.path.join(COLAB_ROOT, "Wan2GP")

if not os.path.isdir(os.path.join(WAN2GP_DIR, ".git")):
    subprocess.run(["git", "clone", "https://github.com/DeepBeepMeep/Wan2GP.git", WAN2GP_DIR], check=True)
else:
    print(f"✅ Wan2GP already exists: {WAN2GP_DIR}")

subprocess.run([
    sys.executable, "-m", "pip", "install", "--timeout", "120", "--retries", "5", "-q",
    "-r", os.path.join(WAN2GP_DIR, "requirements.txt")
], check=True)
subprocess.run([
    sys.executable, "-m", "pip", "install", "--timeout", "120", "--retries", "5", "-q",
    "mmgp", "gradio"
], check=True)
print("✅ Dependencies installed")


✅ Dependencies installed


## 📥 Cell 3 — Download Models

In [3]:
# Cell 3: Download All Required Models
import os
import shutil
from huggingface_hub import hf_hub_download

REPO = "DeepBeepMeep/TTS"
COLAB_ROOT = os.environ.get("AUDIOGEN_ROOT", "/content")
MODEL_DIR = os.path.join(COLAB_ROOT, "Wan2GP", "models")
os.makedirs(MODEL_DIR, exist_ok=True)

def download_if_missing(filename):
    destination = os.path.join(MODEL_DIR, filename)
    if os.path.exists(destination):
        print(f"  ✓ Already exists: {filename}")
        return
    print(f"Downloading {filename}...")
    hf_hub_download(repo_id=REPO, filename=filename, local_dir=MODEL_DIR)
    print(f"  ✓ {filename}")

# Direct local_dir downloads avoid platform-specific temporary paths and symlinks.
download_if_missing("ace_step_v1_5_xl_transformer_quanto_bf16_int8.safetensors")
download_if_missing("ace_step15/ace_step_v1_5_audio_vae_bf16.safetensors")
download_if_missing("ace_step15/silence_latent.pt")

# Text encoder used by the Wan2GP AceStep handler.
QWEN3_FOLDER = "Qwen3-Embedding-0.6B"
for filename in [
    "model.safetensors", "config.json", "tokenizer.json",
    "tokenizer_config.json", "special_tokens_map.json",
]:
    download_if_missing(os.path.join(QWEN3_FOLDER, filename))

for cache_dir in [os.path.join(MODEL_DIR, ".cache")]:
    if os.path.exists(cache_dir):
        shutil.rmtree(cache_dir)

print(f"✅ Model files are ready under {MODEL_DIR}")


/usr/local/lib/python3.13/dist-packages/huggingface_hub/utils/_auth.py:104: UserWarning: 
Error while fetching `HF_TOKEN` secret value from your vault: 'Requesting secret HF_TOKEN timed out. Secrets can only be fetched when running from the Colab UI.'.
You are not authenticated with the Hugging Face Hub in this notebook.
If the error persists, please let us know by opening an issue on GitHub (https://github.com/huggingface/huggingface_hub/issues/new).
  warnings.warn(


ace_step_v1_5_xl_transformer_quanto_bf16(…):   0%|          | 0.00/5.51G [00:00<?, ?B/s]

  ✓ ace_step_v1_5_xl_transformer_quanto_bf16_int8.safetensors


ace_step15/ace_step_v1_5_audio_vae_bf16.(…):   0%|          | 0.00/337M [00:00<?, ?B/s]

  ✓ ace_step15/ace_step_v1_5_audio_vae_bf16.safetensors


...

  ✓ Qwen3-Embedding-0.6B/tokenizer.json


tokenizer_config.json: 0.00B [00:00, ?B/s]

  ✓ Qwen3-Embedding-0.6B/tokenizer_config.json


special_tokens_map.json:   0%|          | 0.00/613 [00:00<?, ?B/s]

  ✓ Qwen3-Embedding-0.6B/special_tokens_map.json
✅ Model files are ready under /content/Wan2GP/models


## 📝 Cell 4 — Write Gradio App Script

In [ ]:
%%writefile run_acestep_xl.py
import gc
import os
import sys
import random
import tempfile
import traceback
import numpy as np
import subprocess
import psutil

# ---- Bootstrap Wan2GP ----
WAN2GP_DIR = os.environ.get("WAN2GP_DIR", "/content/Wan2GP")
if not os.path.isdir(WAN2GP_DIR):
    raise FileNotFoundError(f"Wan2GP directory not found: {WAN2GP_DIR}")
sys.path.insert(0, WAN2GP_DIR)
os.chdir(WAN2GP_DIR)
os.environ["PYTORCH_ALLOC_CONF"] = "expandable_segments:True,max_split_size_mb:128,garbage_collection_threshold:0.5"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

import torch
import gradio as gr

# ==== GPU INFO ====
if not torch.cuda.is_available():
    raise RuntimeError("CUDA GPU is required. In Colab select Runtime → Change runtime type → GPU.")
print(f"GPU: {torch.cuda.get_device_name()}")
print(f"Compute Capability: {torch.cuda.get_device_capability()}")
print(f"VRAM: {torch.cuda.get_device_properties(0).total_memory / 1024**3:.1f} GB")
ram = psutil.virtual_memory()
print(f"RAM: {ram.total / 1024**3:.1f} GB total, {ram.available / 1024**3:.1f} GB available")
sys.stdout.flush()

# ==== Force attention backends for T4 (SM 7.5, no flash attn) ====
torch.backends.cuda.enable_flash_sdp(False)
torch.backends.cuda.enable_mem_efficient_sdp(True)
torch.backends.cuda.enable_math_sdp(True)

# ==== LOAD MODEL VIA WAN2GP ====
print("\nLoading AceStep 1.5 XL Turbo (quanto int8)...")
sys.stdout.flush()

from mmgp import offload
from shared.utils import files_locator as fl

fl.set_checkpoints_paths(["models", "ckpts", "."])

from models.TTS.ace_step_handler import family_handler

base_model_type = "ace_step_v1_5_xl"
model_def = {
    "ace_step15_transformer_variant": "xl_turbo",
    "text_encoder_folder": "acestep-5Hz-lm-1.7B",
}
extra = family_handler.query_model_def(base_model_type, model_def)
model_def.update(extra)

transformer_path = os.path.join("models", "ace_step_v1_5_xl_transformer_quanto_bf16_int8.safetensors")
if not os.path.isfile(transformer_path):
    raise FileNotFoundError(f"Transformer not found at {transformer_path}")
print(f"  Transformer: {os.path.basename(transformer_path)}")
sys.stdout.flush()

ace_model, pipe = family_handler.load_model(
    model_filename=transformer_path,
    model_type="ace_step_v1_5_xl",
    base_model_type=base_model_type,
    model_def=model_def,
    dtype=torch.bfloat16,
    VAE_dtype=torch.float32,
    text_encoder_filename=None,
)

# ==== Verify pipeline components ====
print("\n--- Pipeline Components ---")
pipe_inner = pipe.get("pipe", pipe) if isinstance(pipe, dict) and "pipe" in pipe else pipe
for name, component in pipe_inner.items():
    if component is not None:
        ctype = type(component).__name__
        print(f"  {name}: {ctype}")
    else:
        print(f"  {name}: None")
sys.stdout.flush()

# ==== Apply mmgp — higher VRAM budgets for T4 15GB ====
print("\nApplying mmgp custom profile (pinned + async + HIGH budget)...")
sys.stdout.flush()

offload.profile(
    pipe_inner,
    profile_no=2,
    quantizeTransformer=False,
    convertWeightsFloatTo=torch.bfloat16,
    pinnedMemory=True,
    asyncTransfers=True,
    budgets={
        "transformer":  10000,
        "text_encoder_2": 2500,
        "codec":         2000,
        "*":             1000,
    },
)
print("✅ mmgp offloading ready (high VRAM budget)!")
sys.stdout.flush()

offload.shared_state["_attention"] = "sdpa"

# ==== Fix: re-promote XL tokenizer quantizer to float32 ====
try:
    quantizer = ace_model.ace_step_transformer.tokenizer.quantizer
    quantizer.float()
    print("✅ XL tokenizer quantizer re-promoted to float32")
except Exception as e:
    print(f"⚠️ Could not re-promote quantizer: {e}")
sys.stdout.flush()

print("\n✅ AceStep 1.5 XL Turbo loaded! Ready to generate music.")
sys.stdout.flush()

# ==== CONSTANTS ====
KEYSCALE_OPTIONS = [
    "", "C major", "C minor", "C# major", "C# minor", "Cb major", "Cb minor",
    "D major", "D minor", "D# major", "D# minor", "Db major", "Db minor",
    "E major", "E minor", "E# major", "E# minor", "Eb major", "Eb minor",
    "F major", "F minor", "F# major", "F# minor", "Fb major", "Fb minor",
    "G major", "G minor", "G# major", "G# minor", "Gb major", "Gb minor",
    "A major", "A minor", "A# major", "A# minor", "Ab major", "Ab minor",
    "B major", "B minor", "B# major", "B# minor", "Bb major", "Bb minor",
]

LANGUAGE_OPTIONS = [
    "", "en", "zh", "ja", "ko", "fr", "de", "es", "pt", "it", "ru",
    "ar", "hi", "th", "vi", "id", "tr", "pl", "nl", "sv", "fi",
    "da", "no", "cs", "ro", "hu", "el", "he", "fa", "ur", "bn",
    "ta", "te", "unknown",
]

AUDIO_TASK_OPTIONS = [
    "Text (Lyrics) → Audio",
    "Cover Mode of Source Audio",
    "Transfer Reference Audio Timbre",
    "Cover + Transfer Timbre",
]

# ==== SINGLE GENERATION (unchanged core logic) ====
@torch.inference_mode()
def _generate_single(
    lyrics, caption, duration, seed,
    num_steps, guidance_scale, temperature,
    bpm, keyscale, timesignature, language,
    negative_prompt, audio_scale, shift, infer_method,
    audio_ref, audio_ref_timbre, audio_task,
    progress_fn=None, progress_offset=0.0, progress_scale=1.0,
    track_label="",
):
    gc.collect()
    torch.cuda.empty_cache()
    torch.cuda.synchronize()

    if seed is None or seed < 0:
        seed = random.randint(0, 2**32 - 1)
    seed = int(seed)

    duration = max(5, min(360, int(duration)))
    num_steps = max(1, int(num_steps))

    custom_settings = {}
    if bpm is not None and str(bpm).strip():
        try:
            bpm_val = int(float(bpm))
            if 30 <= bpm_val <= 300:
                custom_settings["bpm"] = bpm_val
        except (ValueError, TypeError):
            pass
    if keyscale and str(keyscale).strip():
        custom_settings["keyscale"] = str(keyscale).strip()
    if timesignature and str(timesignature).strip():
        try:
            ts_val = int(float(timesignature))
            if ts_val in (2, 3, 4, 6):
                custom_settings["timesignature"] = ts_val
        except (ValueError, TypeError):
            pass
    if language and str(language).strip():
        custom_settings["language"] = str(language).strip().lower()

    audio_guide = None
    audio_guide2 = None
    audio_prompt_type = ""

    task = str(audio_task or "")
    if "Cover" in task and "Timbre" in task:
        audio_prompt_type = "AB"
    elif "Cover" in task:
        audio_prompt_type = "A"
    elif "Timbre" in task:
        audio_prompt_type = "B"

    if "A" in audio_prompt_type and audio_ref is not None and os.path.isfile(str(audio_ref)):
        audio_guide = str(audio_ref)
    elif "A" in audio_prompt_type:
        return None, "❌ Cover Mode requires a Source Audio file."

    if "B" in audio_prompt_type and audio_ref_timbre is not None and os.path.isfile(str(audio_ref_timbre)):
        audio_guide2 = str(audio_ref_timbre)
    elif "B" in audio_prompt_type:
        return None, "❌ Timbre Transfer requires a Timbre Reference file."

    neg = str(negative_prompt or "").strip()
    if not neg:
        neg = "NO USER INPUT"

    free_vram = torch.cuda.mem_get_info()[0] / 1024**3
    ram = psutil.virtual_memory()
    print(f"\n{'='*60}")
    print(f"{track_label} | Task: {audio_task}")
    print(f"Generating: {duration}s, {num_steps} steps, seed={seed}")
    print(f"Lyrics: {lyrics[:80]}...")
    print(f"Caption: {caption[:80]}...")
    if custom_settings:
        print(f"Custom: {custom_settings}")
    print(f"  VRAM free: {free_vram:.2f} GB | RAM free: {ram.available / 1024**3:.1f} GB")
    print(f"{'='*60}")
    sys.stdout.flush()

    total_steps = [num_steps]
    current_step = [0]

    def cb(step_idx=-1, latent=None, force_refresh=True, read_state=False,
           override_num_inference_steps=-1, pass_no=-1, preview_meta=None,
           denoising_extra="", progress_unit=None, **kwargs):
        if override_num_inference_steps is not None and override_num_inference_steps > 0:
            total_steps[0] = override_num_inference_steps
        if step_idx is not None and step_idx >= 0:
            current_step[0] = step_idx + 1
            frac = current_step[0] / max(total_steps[0], 1)
            free_v = torch.cuda.mem_get_info()[0] / 1024**3
            print(f"  {track_label} Step {current_step[0]}/{total_steps[0]} | VRAM free: {free_v:.2f} GB")
            sys.stdout.flush()
            if progress_fn is not None:
                scaled = progress_offset + min(frac * 0.9, 0.95) * progress_scale
                progress_fn(min(scaled, 0.99), desc=f"{track_label}: Step {current_step[0]}/{total_steps[0]}")

    result = ace_model.generate(
        lyrics,
        0,
        audio_guide,
        alt_prompt=caption if caption and caption.strip() else None,
        audio_guide2=audio_guide2,
        audio_prompt_type=audio_prompt_type,
        temperature=float(temperature),
        duration_seconds=duration,
        num_inference_steps=num_steps,
        guidance_scale=float(guidance_scale),
        seed=seed,
        custom_settings=custom_settings if custom_settings else None,
        lm_negative_prompt=neg,
        audio_scale=float(audio_scale),
        shift=float(shift),
        infer_method=str(infer_method),
        callback=cb,
    )

    if result is None:
        return None, "❌ Generation failed (returned None)."

    audio_data = None
    audio_sr = 48000

    if isinstance(result, dict):
        audio_data = result.get("x")
        if audio_data is None:
            audio_data = result.get("audio")
        audio_sr = result.get("audio_sampling_rate", result.get("sample_rate", 48000))
    elif isinstance(result, tuple):
        audio_data = result[0]
        if len(result) > 1 and isinstance(result[1], int):
            audio_sr = result[1]
    else:
        audio_data = result

    if audio_data is None:
        return None, "❌ No audio data returned."

    if torch.is_tensor(audio_data):
        audio_np = audio_data.cpu().float().numpy()
    elif isinstance(audio_data, np.ndarray):
        audio_np = audio_data
    else:
        return None, f"❌ Unknown audio type: {type(audio_data)}"

    if audio_np.ndim == 3:
        audio_np = audio_np.squeeze(0)
    if audio_np.ndim == 2 and audio_np.shape[0] <= 2:
        audio_np = audio_np.T

    import soundfile as sf
    wav_path = tempfile.mktemp(suffix=f"_seed{seed}.wav")
    sf.write(wav_path, audio_np, int(audio_sr))

    mp3_path = wav_path.replace(".wav", ".mp3")
    try:
        subprocess.run([
            "ffmpeg", "-y", "-i", wav_path,
            "-codec:a", "libmp3lame", "-b:a", "192k", mp3_path
        ], check=True, capture_output=True)
        out_path = mp3_path if os.path.exists(mp3_path) and os.path.getsize(mp3_path) > 0 else wav_path
    except Exception:
        out_path = wav_path

    del audio_data, audio_np
    gc.collect()
    torch.cuda.empty_cache()

    return out_path, seed


# ==== MULTI-TRACK WRAPPER — uses gr.update() to only touch active tracks ====
@torch.inference_mode()
def generate_music(
    lyrics, caption, duration, seed,
    num_steps, guidance_scale, temperature,
    bpm, keyscale, timesignature, language,
    negative_prompt, audio_scale, shift, infer_method,
    audio_ref, audio_ref_timbre, audio_task, num_tracks,
    progress=gr.Progress(),
):
    try:
        n = max(1, min(4, int(num_tracks)))
        base_seed = int(seed) if seed is not None and int(seed) >= 0 else random.randint(0, 2**32 - 1)

        outputs = [None, None, None, None]
        statuses = []

        for i in range(n):
            track_seed = base_seed + i
            label = f"Track {i+1}/{n}"
            progress(i / n, desc=f"{label}: starting...")
            print(f"\n>>> {label} (seed={track_seed})")
            sys.stdout.flush()

            try:
                result_path, used_seed = _generate_single(
                    lyrics, caption, duration, track_seed,
                    num_steps, guidance_scale, temperature,
                    bpm, keyscale, timesignature, language,
                    negative_prompt, audio_scale, shift, infer_method,
                    audio_ref, audio_ref_timbre, audio_task,
                    progress_fn=progress,
                    progress_offset=i / n,
                    progress_scale=1.0 / n,
                    track_label=label,
                )
                if result_path is None:
                    statuses.append(f"❌ {label}: {used_seed}")
                else:
                    outputs[i] = result_path
                    statuses.append(f"✅ {label}: seed={used_seed}")
                    print(f"  ✅ {label} saved: {result_path}")
                    sys.stdout.flush()
            except Exception as e:
                traceback.print_exc()
                gc.collect()
                torch.cuda.empty_cache()
                statuses.append(f"❌ {label}: {str(e)}")

        progress(1.0, desc="Done!")
        status_text = "\n".join(statuses)

        # Use gr.update() — only set value for tracks we used, hide unused ones
        results = []
        for i in range(4):
            if i < n:
                results.append(gr.update(value=outputs[i], visible=True))
            else:
                results.append(gr.update(value=None, visible=False))
        results.append(status_text)
        return results

    except Exception as e:
        traceback.print_exc()
        gc.collect()
        torch.cuda.empty_cache()
        return [gr.update(value=None, visible=False) for _ in range(4)] + [f"❌ Error: {str(e)}"]


# ==== GRADIO UI ====
CSS = """
@import url('https://fonts.googleapis.com/css2?family=Inter:wght@400;600;700&display=swap');
* { font-family: 'Inter', sans-serif !important; }
.gradio-container { max-width: 1000px !important; margin: auto !important; }
.brand-header { text-align: center; background: linear-gradient(135deg, #667eea 0%, #764ba2 100%); padding: 28px; border-radius: 15px; margin-bottom: 20px; box-shadow: 0 10px 25px rgba(102,126,234,0.3); }
.brand-title { color: white; font-size: 2em; font-weight: 700; margin: 0 0 6px 0; }
.brand-subtitle { color: rgba(255,255,255,0.88); font-size: 1em; margin-bottom: 16px; }
.btn-row { display: flex; justify-content: center; gap: 10px; flex-wrap: wrap; }
.social-btn { display: inline-flex; align-items: center; justify-content: center; min-width: 150px; padding: 10px 18px; border-radius: 10px; font-weight: 700; font-size: 13px; text-decoration: none; color: white; white-space: nowrap; }
.yt-btn  { background: #FF0000; box-shadow: 0 4px 12px rgba(255,0,0,0.3); }
.x-btn   { background: #000000; box-shadow: 0 4px 12px rgba(0,0,0,0.25); }
.sup-btn { background: linear-gradient(135deg,#f6d365,#fda085); box-shadow: 0 4px 12px rgba(253,160,133,0.35); }
button.primary { background: linear-gradient(135deg, #667eea 0%, #764ba2 100%) !important; color: white !important; font-weight: 600 !important; border-radius: 12px !important; }
"""

BRAND_HTML = """
<div class="brand-header">
  <div class="brand-title">🎵 AceStep 1.5 XL Turbo (4B)</div>
  <div class="brand-subtitle">Created by <strong>AIQuest Academy</strong> &nbsp;|&nbsp; Colab GPU · Wan2GP + mmgp · INT8</div>
  <div class="btn-row">
    <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1" target="_blank" class="social-btn yt-btn">▶ Subscribe</a>
    <a href="https://x.com/aiquestacademy" target="_blank" class="social-btn x-btn">𝕏 Follow on X</a>
    <a href="https://aiquest.site" target="_blank" class="social-btn sup-btn">❤️ Support My Work</a>
  </div>
</div>
"""

with gr.Blocks(css=CSS, theme=gr.themes.Soft(), title="AceStep 1.5 XL Turbo") as demo:
    gr.HTML(BRAND_HTML)

    with gr.Row():
        with gr.Column(scale=3):
            audio_task = gr.Dropdown(
                label="🎯 Audio Task",
                choices=AUDIO_TASK_OPTIONS,
                value="Text (Lyrics) → Audio",
            )

            lyrics = gr.Textbox(
                label="🎤 Lyrics / Prompt (Write [Instrumental] for instrumental only)",
                lines=8,
                value="[Verse]\nI wake up every morning, feeling alive\nThe world outside is bright, the sun is on my side\n\n[Chorus]\nWe're dancing through the fire\nRising ever higher\nNothing's gonna stop us now",
                placeholder="[Verse]\nYour lyrics here...\n[Chorus]\n...\n\nUse [Instrumental] for no vocals.",
            )

            caption = gr.Textbox(
                label="🎶 Music Caption (Describe the style, genre, instruments, and mood)",
                lines=3,
                value="Dreamy synth-pop with shimmering pads, soft vocals, and a slow dance groove.",
                placeholder="disco, electric guitar, energetic drums, female vocals...",
            )

            with gr.Row():
                bpm = gr.Textbox(label="🥁 BPM (30-300)", value="", placeholder="e.g. 120 (empty = auto)")
                keyscale = gr.Dropdown(label="🎹 KeyScale", choices=KEYSCALE_OPTIONS, value="", allow_custom_value=True)

            with gr.Row():
                timesignature = gr.Dropdown(label="🎼 Time Signature", choices=["", "2", "3", "4", "6"], value="")
                language = gr.Dropdown(label="🌐 Language (ISO code)", choices=LANGUAGE_OPTIONS, value="", allow_custom_value=True)

            duration = gr.Slider(label="⏱️ Duration (seconds)", minimum=5, maximum=360, value=120, step=1)

            with gr.Accordion("⚙️ Advanced Settings", open=False):
                with gr.Tab("General"):
                    with gr.Row():
                        seed = gr.Number(label="🎲 Seed (-1 = random)", value=-1, precision=0)
                        num_steps = gr.Slider(label="🔄 Inference Steps", minimum=1, maximum=50, value=8, step=1)
                    with gr.Row():
                        guidance_scale = gr.Slider(label="📏 Guidance Scale", minimum=0.0, maximum=15.0, value=7.0, step=0.1)
                        temperature = gr.Slider(label="🌡️ Temperature", minimum=0.0, maximum=2.0, value=1.0, step=0.05)
                    with gr.Row():
                        audio_scale = gr.Slider(label="🎚️ Audio Cover Strength", minimum=0.0, maximum=1.0, value=1.0, step=0.05)
                        shift = gr.Slider(label="📐 Shift", minimum=0.0, maximum=5.0, value=1.0, step=0.1)
                    infer_method = gr.Radio(label="🧮 Inference Method", choices=["ode", "sde"], value="ode")
                    negative_prompt = gr.Textbox(
                        label="🚫 Negative Prompt", lines=2, value="",
                        placeholder="Describe what you DON'T want (empty = none)",
                    )

                with gr.Tab("Audio Reference"):
                    gr.Markdown(
                        "**Cover Mode** → upload Source Audio below.\n\n"
                        "**Transfer Timbre** → upload Timbre Reference below.\n\n"
                        "**Cover + Transfer** → upload both."
                    )
                    audio_ref = gr.Audio(label="🎵 Source Audio (for Cover Mode)", type="filepath")
                    audio_ref_timbre = gr.Audio(label="🎙️ Timbre Reference (for Timbre Transfer)", type="filepath")

            num_tracks = gr.Slider(
                label="🔢 Number of Output Tracks",
                minimum=1, maximum=4, value=1, step=1,
                info="Each track uses seed, seed+1, ... Generated sequentially."
            )

            gen_btn = gr.Button("🎵 Generate Music", variant="primary", size="lg")

        with gr.Column(scale=2):
            status_out = gr.Textbox(label="📊 Status", interactive=False, lines=3)
            audio_out_1 = gr.Audio(label="🎧 Track 1", type="filepath")
            audio_out_2 = gr.Audio(label="🎧 Track 2", type="filepath", visible=False)
            audio_out_3 = gr.Audio(label="🎧 Track 3", type="filepath", visible=False)
            audio_out_4 = gr.Audio(label="🎧 Track 4", type="filepath", visible=False)

    gen_btn.click(
        fn=generate_music,
        inputs=[
            lyrics, caption, duration, seed,
            num_steps, guidance_scale, temperature,
            bpm, keyscale, timesignature, language,
            negative_prompt, audio_scale, shift, infer_method,
            audio_ref, audio_ref_timbre, audio_task, num_tracks,
        ],
        outputs=[audio_out_1, audio_out_2, audio_out_3, audio_out_4, status_out],
    )

print("\nLaunching Gradio...")
sys.stdout.flush()
demo.queue(max_size=1)
demo.launch(
    share=True,
    inline=False,
    debug=False,
    show_error=True,
    max_threads=2,
    ssr_mode=False,
)

## 🚀 Cell 5 — Launch!

In [ ]:
# Cell 5: Launch!
import os
import subprocess
import sys

COLAB_ROOT = os.environ.get("AUDIOGEN_ROOT", "/content")
app_path = os.path.join(COLAB_ROOT, "run_acestep_xl.py")
if not os.path.isfile(app_path):
    raise FileNotFoundError(f"App script not found: {app_path}. Run Cell 4 first.")

subprocess.run([sys.executable, "-u", app_path], cwd=COLAB_ROOT, check=True)


---

<div align="center">

  <a href="https://www.youtube.com/@aiquestacademy?sub_confirmation=1">
    <img src="https://img.shields.io/badge/Subscribe%20on%20YouTube-FF0000?style=for-the-badge&logo=youtube&logoColor=white" />
  </a>
  <a href="https://x.com/aiquestacademy">
    <img src="https://img.shields.io/badge/Follow%20on%20X-000000?style=for-the-badge&logo=x&logoColor=white" />
  </a>
  <a href="https://aiquest.site">
    <img src="https://img.shields.io/badge/Support%20My%20Work-f59e0b?style=for-the-badge&logoColor=white" />
  </a>

</div>

<p align="center" style="color:#6b7280; font-size:12px; margin-top:8px;">
  ⚡ Made with ❤️ by <strong>AIQUEST Academy</strong> · aiquest.site · © All rights reserved
</p>

---